# 1.01 - Simple Completion

---

### Practical AgentOps: From PoC to Prod with MLflow 3 — ODSC AI East 2026

**The scenario:** You're an ML engineer at a large software/AI company that is interested in creating a high quality MLflow assistant. The PM shared a prototype with beta users and everyone likes it — now the pressure is on to turn it into something shippable. That means evaluation.

In this stage we build the baseline agent — a simple completions API call with a system prompt. 

By the end of this notebook you will have hands-on experience with creating a simple completions agent and viewing the tracing in the MLflow UI.

> **No API key?** You can read through every cell and understand the concepts. Cells that call the model are clearly marked — skip those and pick back up at the next markdown section.

---
## 0 — Environment Setup

1. Follow steps in the 'workshop/setup' folder
2. cmd/ctrl + shift + p
3. Python: Select Interpreter
4. Select your recently created environment

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import mlflow

load_dotenv()  # reads GEMINI_API_KEY from .env

/Users/jon/Documents/GitHub/practical-agent-ops-mlflow3/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

### Configuring the OpenAI client to point at Gemini

We use the **OpenAI SDK** pointed at Google's OpenAI-compatible endpoint. This is an increasingly common pattern — the SDK is a familiar interface, and swapping providers is just a matter of changing `base_url` and the API key.

| Provider | `base_url` | Key env var |
|---|---|---|
| OpenAI (default) | *(omit)* | `OPENAI_API_KEY` |
| **Gemini** ✓ | `https://generativelanguage.googleapis.com/v1beta/openai/` | `GEMINI_API_KEY` |
| Azure OpenAI | `https://<resource>.openai.azure.com/` | `AZURE_OPENAI_API_KEY` |
| Ollama (local) | `http://localhost:11434/v1` | *(none required)* |

In [5]:
client = OpenAI(
    api_key=os.environ["GEMINI_API_KEY"],
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

#MODEL = "gemini-3.1-flash-lite-preview"
#Backup model
MODEL = "gemini-3-flash-preview"

---
## 1 — Autologging and the Experiment

Two lines are all it takes to get full tracing into the MLflow UI.

- `mlflow.set_experiment()` — creates (or resumes) a named experiment bucket for all runs
- `mlflow.openai.autolog()` — patches the OpenAI SDK to emit a trace for every `chat.completions.create()` call, capturing inputs, outputs, token counts, and latency automatically

Once these are set, **every call in this notebook is captured with no extra code**.

In [ ]:
tracking_uri = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
mlflow.set_tracking_uri(tracking_uri)

mlflow.set_experiment("mlflow-agent-1")
mlflow.openai.autolog()

print("Autologging enabled. Open the MLflow UI to follow along:")
print("  mlflow server --backend-store-uri sqlite:///mlflow.db --host 127.0.0.1 --port 5000")
print("If previous instance is running, run in terminal: kill -9 $(lsof -t -i:3000)")

2026/04/14 09:20:53 INFO mlflow.tracking.fluent: Experiment with name 'recipe-assistant-1' does not exist. Creating a new experiment.


Autologging enabled. Open the MLflow UI to follow along:
  mlflow server --backend-store-uri sqlite:///mlflow.db --host 127.0.0.1 --port 5000
If previous instance is running, run in terminal: kill -9 $(lsof -t -i:3000)


---
## 2 — First Completions Call

Before we think about evaluation, structure, or scaffolding — let's just build the thing. A system prompt and a single completions call. This is the PoC.

In [ ]:
SYSTEM_PROMPT = """You are a helpful MLflow assistant."""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": "How do I turn on auto logging?"},
    ],
    temperature=0.1,
    max_completion_tokens=512,
)

print(response.choices[0].message.content)

To make perfectly scrambled eggs—creamy, soft, and flavorful—the secret lies in **low heat** and **patience**. 

Here is the step-by-step guide to achieving restaurant-quality eggs at home.

### 1. The Ingredients
*   **Eggs:** Use the freshest eggs possible. (2–3 per person).
*   **Fat:** Unsalted butter is best for flavor and texture.
*   **Salt:** Fine sea salt or kosher salt.
*   **Optional:** A splash of heavy cream or whole milk (for extra richness), though many chefs prefer eggs alone.

###


Trace(trace_id=tr-a166ddabb66eeebc6354f2713eee77db)

### Ship it! ...right?

The completions call works fine so far.

> **Sam (Product):** "How do we *know* it's good? Like, do we have any metrics? What if it gives someone outdated functions or design patterns? What if the next model update makes it worse? Have you discussed anything with legal? When can we start testing?"

You don't have an answer yet. Time to think about **evaluation**.